# 🚀 Q&A System V2 - Clean Modular Interface

Refactored UI with extracted modules for better maintainability.

In [1]:
# Setup and imports
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

# Import core system setup
from qanda_module.config import setup_system
from qanda_module.ui_gradio import launch_ui_with_toggle

print("📦 Imports successful!")

📦 Imports successful!


In [3]:
# System setup
print("🚀 Setting up system...")

helpers, qa_chain, config = setup_system("../config.yaml")

print("✅ System setup complete!")
print(f"Model: {config.chat_model_name}")
print(f"Temperature: {config.temperature}")
print(f"Debug: {config.debug_enabled}")

🚀 Setting up system...
⚠️  Some modules not yet migrated: No module named 'langchain_community'
You'll need to copy over legacy modules or install dependencies
✅ System setup complete!
Model: gpt-4o-mini
Temperature: 0.3
Debug: False


In [5]:
# UI Selection and Launch
USE_NEW_DESIGN = True  # Toggle between current and new UI

demo = launch_ui_with_toggle(
    helpers, 
    qa_chain, 
    config, 
    design="new" if USE_NEW_DESIGN else "current"
)

print(f"🎯 Demo created using {'NEW' if USE_NEW_DESIGN else 'CURRENT'} design")

🎨 Creating new UI...
Loaded 1035 panelists using date-based latest episode logic


AttributeError: 'NoneType' object has no attribute 'panelist_lookup'

In [4]:
# Launch the interface
print("🌐 Launching Gradio interface...")

demo.launch(
    share=True,  # Set to True for public link
    server_port=7860,
    show_error=True,
    debug=True  # Shows more info in console
)

🌐 Launching Gradio interface...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://3d761e8dc0c5c14e20.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


🔍 Processing question: What are CHRISTOPHER PYNE's most notable views?
🔍 Parameters: k=80, style=Balanced
Processing: What are CHRISTOPHER PYNE's most notable views?...
✅ Question processed successfully


/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/swang/workspace/repos/qanda_clean1/qanda-v2/.venv/lib/python3.12/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3d761e8dc0c5c14e20.gradio.live


In [ ]:
# Optional: Quick testing and experiments

def quick_test():
    """Test the system with a simple query."""
    from qanda_module.ui_gradio import handle_question
    
    test_question = "What did panelists say about climate change?"
    result = handle_question(test_question, 10, "Concise", helpers, qa_chain, config)
    
    print(f"✅ Test completed: {result[2]}")
    print(f"Answer length: {len(result[0])} chars")
    return result

# Uncomment to run test:
# test_result = quick_test()

In [ ]:
# Stop the demo when needed
# demo.close()

In [6]:
# Create database connection
import duckdb

# Using your config (assuming you already have helpers, qa_chain, config loaded)
con = duckdb.connect(config.duck_db_name)

# Or directly with the path if you prefer:
# con = duckdb.connect("../data/qanda_v2.duckdb")

In [7]:
# Check if we're getting the right "latest" entries
con.execute("""
    SELECT name, id, profession, link, 
           ROW_NUMBER() OVER (PARTITION BY name ORDER BY id DESC) as rn
    FROM dim_panellist 
    WHERE name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY name, id DESC
""").df()

,name,id,profession,link,rn
0,Christopher Pyne,2933,Opposition education spokesman,https://www.abc.net.au/qanda/christopher-pyne/...,1
1,Christopher Pyne,2753,shadow education minister,https://www.abc.net.au/qanda/christopher-pyne/...,2
2,Christopher Pyne,2609,Oppostion education spokesman,https://www.abc.net.au/qanda/christopher-pyne/...,3
3,Christopher Pyne,2492,shadow education minister,https://www.abc.net.au/qanda/christopher-pyne/...,4
4,Christopher Pyne,2447,shadow education minister,https://www.abc.net.au/qanda/christopher-pyne/...,5
5,Christopher Pyne,2383,shadow education minister,https://www.abc.net.au/qanda/christopher-pyne/...,6
6,Christopher Pyne,2331,Opposition education spokesperson,https://www.abc.net.au/qanda/christopher-pyne/...,7
7,Christopher Pyne,2228,Opposition education spokesperson,https://www.abc.net.au/qanda/christopher-pyne/...,8
8,Christopher Pyne,2165,Shadow Education Minister,https://www.abc.net.au/qanda/christopher-pyne/...,9
9,Christopher Pyne,2099,Shadow Education Minister,https://www.abc.net.au/qanda/christopher-pyne/...,10


In [8]:
# Check panelist entries
validation_result = con.execute("""
    SELECT name, id, profession, link, 
           ROW_NUMBER() OVER (PARTITION BY name ORDER BY id DESC) as rn
    FROM dim_panellist 
    WHERE name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY name, id DESC
""").df()

print("Panelist entries (newest first by ID):")
print(validation_result)

# Check for duplicates
duplicates = con.execute("""
    SELECT name, COUNT(*) as count 
    FROM dim_panellist 
    WHERE name IS NOT NULL 
    GROUP BY name 
    HAVING COUNT(*) > 1 
    ORDER BY count DESC 
    LIMIT 10
""").df()

print(f"\nFound {len(duplicates)} panelists with multiple entries:")
print(duplicates)

Panelist entries (newest first by ID):
                name    id                                 profession  \
0   Christopher Pyne  2933             Opposition education spokesman   
1   Christopher Pyne  2753                  shadow education minister   
2   Christopher Pyne  2609              Oppostion education spokesman   
3   Christopher Pyne  2492                  shadow education minister   
4   Christopher Pyne  2447                  shadow education minister   
5   Christopher Pyne  2383                  shadow education minister   
6   Christopher Pyne  2331          Opposition education spokesperson   
7   Christopher Pyne  2228          Opposition education spokesperson   
8   Christopher Pyne  2165                  Shadow Education Minister   
9   Christopher Pyne  2099                  Shadow Education Minister   
10  Christopher Pyne  2044                  Shadow Education Minister   
11  Christopher Pyne  2009                  Shadow Education Minister   
12  Christop

In [9]:
# Check using actual episode dates instead of IDs
date_validation = con.execute("""
    WITH panelist_episodes AS (
        SELECT DISTINCT
            fr.speaker_name,
            dp.id as panelist_id,
            dp.profession,
            dp.link,
            de.date as episode_date
        FROM fact_responses fr
        JOIN dim_panellist dp ON fr.speaker_name = dp.name
        JOIN dim_episode de ON fr.episode_id = de.id
        WHERE fr.speaker_type = 3 
        AND fr.speaker_name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ),
    latest_by_date AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY speaker_name ORDER BY episode_date DESC, panelist_id DESC) as rn
        FROM panelist_episodes
    )
    SELECT speaker_name, panelist_id, profession, episode_date, rn
    FROM latest_by_date
    WHERE speaker_name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY speaker_name, episode_date DESC, panelist_id DESC
""").df()

print("Panelist entries ordered by actual episode dates:")
print(date_validation)

# Check which entry is "latest" by date vs by ID
latest_by_date = date_validation[date_validation['rn'] == 1]
print("\nLatest entries by episode date:")
print(latest_by_date[['speaker_name', 'panelist_id', 'profession', 'episode_date']])

Panelist entries ordered by actual episode dates:
Empty DataFrame
Columns: [speaker_name, panelist_id, profession, episode_date, rn]
Index: []

Latest entries by episode date:
Empty DataFrame
Columns: [speaker_name, panelist_id, profession, episode_date]
Index: []


In [10]:
df_test = con.execute("""
    SELECT * FROM dim_panellist LIMIT 10
""").df()

In [12]:
df_test.head().T


,0,1,2,3,4
date,2009-04-09,2009-04-09,2009-04-09,2009-04-09,2009-04-09
title,"Banks, Bikies and Broadband","Banks, Bikies and Broadband","Banks, Bikies and Broadband","Banks, Bikies and Broadband","Banks, Bikies and Broadband"
url,https://www.abc.net.au/qanda/banks-bikies-and-...,https://www.abc.net.au/qanda/banks-bikies-and-...,https://www.abc.net.au/qanda/banks-bikies-and-...,https://www.abc.net.au/qanda/banks-bikies-and-...,https://www.abc.net.au/qanda/banks-bikies-and-...
host,Tony Jones,Tony Jones,Tony Jones,Tony Jones,Tony Jones
id,2889,2890,2891,2887,2888
name,John Hewson,Jane Caro,Andrew Boe,Tony Burke,Helen Coonan
alias,None,None,None,None,None
profession,former Liberal leader,social commentator,lawyer,Minister for Agriculture,shadow finance minister
link,https://www.abc.net.au/qanda/john-hewson/10644262,https://www.abc.net.au/qanda/jane-caro/10644226,https://www.abc.net.au/qanda/andrew-boe/10644260,https://www.abc.net.au/qanda/tony-burke/10644258,https://www.abc.net.au/qanda/helen-coonan/1064...
profile,Dr John Hewson came to national prominence as ...,"Jane Caro wears many hats; including author, l...",Andrew Boe is a lawyer in private practice in ...,"Tony Burke is the Minister for Agriculture, Fi...",Helen Coonan entered the Senate at the 1996 el...


In [13]:
# Check if speaker names match between tables
name_check = con.execute("""
    SELECT DISTINCT fr.speaker_name as response_name,
           dp.name as panelist_name
    FROM fact_responses fr
    FULL OUTER JOIN dim_panellist dp ON fr.speaker_name = dp.name
    WHERE fr.speaker_name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
       OR dp.name IN ('Christopher Pyne', 'Malcolm Turnbull', 'Penny Wong')
    ORDER BY response_name, panelist_name
""").df()

print("Name matching check:")
print(name_check)

Name matching check:
  response_name     panelist_name
0          None  Christopher Pyne
1          None  Malcolm Turnbull
2          None        Penny Wong


In [14]:
# See what speaker names exist in fact_responses
actual_names = con.execute("""
    SELECT DISTINCT speaker_name, COUNT(*) as count
    FROM fact_responses 
    WHERE speaker_type = 3
    AND (UPPER(speaker_name) LIKE '%PYNE%' 
         OR UPPER(speaker_name) LIKE '%TURNBULL%' 
         OR UPPER(speaker_name) LIKE '%WONG%')
    GROUP BY speaker_name
    ORDER BY count DESC
""").df()

print("Actual speaker names in fact_responses:")
print(actual_names)

Actual speaker names in fact_responses:
       speaker_name  count
0  CHRISTOPHER PYNE   1552
1  MALCOLM TURNBULL   1251
2        PENNY WONG    835
3   CHRISOPHER PYNE     60
4     LUCY TURNBULL     37
5    CHRISTINE WONG     17


In [15]:
# Try case-insensitive match
case_insensitive_check = con.execute("""
    SELECT DISTINCT 
        fr.speaker_name as response_name,
        dp.name as panelist_name,
        UPPER(fr.speaker_name) as response_upper,
        UPPER(dp.name) as panelist_upper
    FROM fact_responses fr
    FULL OUTER JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE UPPER(fr.speaker_name) LIKE '%PYNE%' 
       OR UPPER(fr.speaker_name) LIKE '%TURNBULL%' 
       OR UPPER(fr.speaker_name) LIKE '%WONG%'
       OR UPPER(dp.name) LIKE '%PYNE%'
       OR UPPER(dp.name) LIKE '%TURNBULL%'
       OR UPPER(dp.name) LIKE '%WONG%'
    ORDER BY response_name, panelist_name
""").df()

print("\nCase-insensitive matching:")
print(case_insensitive_check)


Case-insensitive matching:
                      response_name     panelist_name  \
0                       ADRIAN WONG              None   
1                          BEN WONG              None   
2                     BENJAMIN WONG              None   
3                   CHRISOPHER PYNE              None   
4                    CHRISTINE WONG    Christine Wong   
5                  CHRISTOPHER PYNE  Christopher Pyne   
6                        EMILY WONG              None   
7                   HANNAH TURNBULL              None   
8                        JAMES WONG              None   
9                     LUCY TURNBULL     Lucy Turnbull   
10                 MALCOLM TURNBULL  Malcolm Turnbull   
11                       PENNY WONG        Penny Wong   
12  R-U DOUBLE D & MALCOLM TURNBULL              None   

                     response_upper    panelist_upper  
0                       ADRIAN WONG              None  
1                          BEN WONG              None  
2    

In [16]:
# Check why panelists don't have matches in dim_panellist
non_matches = con.execute("""
    SELECT DISTINCT 
        fr.speaker_name,
        COUNT(*) as response_count
    FROM fact_responses fr
    LEFT JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE fr.speaker_type = 3 
    AND dp.name IS NULL  -- No match found
    GROUP BY fr.speaker_name
    ORDER BY response_count DESC
    LIMIT 20
""").df()

print("Panelists in fact_responses with NO match in dim_panellist:")
print(non_matches)

Panelists in fact_responses with NO match in dim_panellist:
                speaker_name  response_count
0             MARK COLERIDGE             127
1                PJ O'ROURKE             115
2               SANDY GUTMAN             107
3                GEORGE PELL              80
4           SUSAN GREENFIELD              68
5            CHRISOPHER PYNE              60
6            JORDAN PETERSON              55
7               CLARE O’NEIL              55
8                ANN SUMMERS              52
9                CORNEL WEST              51
10                 CINDY PAN              49
11             KERRY O’BRIEN              44
12  CONNIE FIERRAVANTI-WELLS              41
13         GRAEME RICHARDSON              39
14             FUZZY AGOLLEY              38
15      PETER HOLMES A COURT              38
16           FELICITY HAMPEL              38
17           MOHAMAD ABDALLA              37
18              MATT COLWELL              37
19            PETER COSGROVE            

In [17]:
# Compare matches vs non-matches for panelists
match_stats = con.execute("""
    SELECT 
        CASE WHEN dp.name IS NOT NULL THEN 'MATCHED' ELSE 'NO_MATCH' END as match_status,
        COUNT(DISTINCT fr.speaker_name) as unique_speakers,
        COUNT(*) as total_responses
    FROM fact_responses fr
    LEFT JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE fr.speaker_type = 3
    GROUP BY match_status
""").df()

print("\nMatch statistics for panelists:")
print(match_stats)


Match statistics for panelists:
  match_status  unique_speakers  total_responses
0      MATCHED             1035           436670
1     NO_MATCH              117             2107


In [18]:
# Check for potential typos like "CHRISOPHER PYNE"
typo_check = con.execute("""
    SELECT DISTINCT fr.speaker_name
    FROM fact_responses fr
    WHERE fr.speaker_type = 3
    AND (fr.speaker_name LIKE '%PYNE%' OR fr.speaker_name LIKE '%TURNBULL%')
    ORDER BY fr.speaker_name
""").df()

print("\nAll PYNE/TURNBULL variants in fact_responses:")
print(typo_check)


All PYNE/TURNBULL variants in fact_responses:
       speaker_name
0   CHRISOPHER PYNE
1  CHRISTOPHER PYNE
2     LUCY TURNBULL
3  MALCOLM TURNBULL


In [19]:
# Get unmatched speakers with their frequency
unmatched_speakers = con.execute("""
    SELECT DISTINCT 
        fr.speaker_name as unmatched_name,
        COUNT(*) as response_count
    FROM fact_responses fr
    LEFT JOIN dim_panellist dp ON UPPER(fr.speaker_name) = UPPER(dp.name)
    WHERE fr.speaker_type = 3 
    AND dp.name IS NULL
    GROUP BY fr.speaker_name
    ORDER BY response_count DESC
""").df()

# Get all available panelist names
available_names = con.execute("""
    SELECT DISTINCT name as available_name
    FROM dim_panellist
    ORDER BY name
""").df()

print(f"Found {len(unmatched_speakers)} unmatched speakers")
print(f"Found {len(available_names)} available panelist names")

Found 117 unmatched speakers
Found 1094 available panelist names


In [20]:
from difflib import SequenceMatcher
import pandas as pd

def similarity(a, b):
    """Calculate similarity ratio between two strings"""
    return SequenceMatcher(None, a.upper(), b.upper()).ratio()

# Find likely matches for top unmatched speakers
likely_matches = []

for _, unmatched_row in unmatched_speakers.head(20).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Calculate similarity with all available names
    matches = []
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        sim_score = similarity(unmatched, available)
        
        if sim_score > 0.6:  # Only show decent matches
            matches.append({
                'unmatched': unmatched,
                'response_count': count,
                'potential_match': available,
                'similarity': round(sim_score, 3)
            })
    
    # Sort by similarity and keep top 3
    matches.sort(key=lambda x: x['similarity'], reverse=True)
    likely_matches.extend(matches[:3])

# Convert to DataFrame and display
matches_df = pd.DataFrame(likely_matches)
print("\nLikely matches (similarity > 0.6):")
print(matches_df.head(30))


Likely matches (similarity > 0.6):
           unmatched  response_count            potential_match  similarity
0     MARK COLERIDGE             127  Archbishop Mark Coleridge       0.718
1     MARK COLERIDGE             127              Mark Carnegie       0.667
2     MARK COLERIDGE             127                Mark Bouris       0.640
3        PJ O'ROURKE             115              P.J. O'Rourke       0.917
4       SANDY GUTMAN             107                 Dan Sultan       0.636
5       SANDY GUTMAN             107                 Stan Grant       0.636
6       SANDY GUTMAN             107             Sandy Plunkett       0.615
7        GEORGE PELL              80       Cardinal George Pell       0.710
8        GEORGE PELL              80                Grace Kelly       0.636
9        GEORGE PELL              80                Jeremy Bell       0.636
10  SUSAN GREENFIELD              68  Baroness Susan Greenfield       0.780
11  SUSAN GREENFIELD              68               B

In [21]:
# Also check for substring matches (different approach)
substring_matches = []

for _, unmatched_row in unmatched_speakers.head(20).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Split into words for partial matching
    unmatched_words = set(unmatched.upper().split())
    
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        available_words = set(available.upper().split())
        
        # Check for common words
        common_words = unmatched_words.intersection(available_words)
        if len(common_words) > 0:
            match_ratio = len(common_words) / max(len(unmatched_words), len(available_words))
            
            if match_ratio > 0.5:  # At least 50% word overlap
                substring_matches.append({
                    'unmatched': unmatched,
                    'response_count': count,
                    'potential_match': available,
                    'common_words': ', '.join(common_words),
                    'word_match_ratio': round(match_ratio, 3)
                })

substring_df = pd.DataFrame(substring_matches)
print("\nSubstring/word matches:")
print(substring_df.head(20))


Substring/word matches:
              unmatched  response_count            potential_match  \
0        MARK COLERIDGE             127  Archbishop Mark Coleridge   
1           GEORGE PELL              80       Cardinal George Pell   
2      SUSAN GREENFIELD              68  Baroness Susan Greenfield   
3       JORDAN PETERSON              55         Dr Jordan Peterson   
4           CORNEL WEST              51             Dr Cornel West   
5             CINDY PAN              49               Dr Cindy Pan   
6       FELICITY HAMPEL              38      Judge Felicity Hampel   
7  PETER HOLMES A COURT              38       Peter Holmes à Court   
8       MOHAMAD ABDALLA              37         Dr Mohamad Abdalla   
9        PETER COSGROVE              34     General Peter Cosgrove   

           common_words  word_match_ratio  
0       COLERIDGE, MARK             0.667  
1          GEORGE, PELL             0.667  
2     SUSAN, GREENFIELD             0.667  
3      JORDAN, PETERSON     

In [22]:
# Enhanced substring matching with better logic
enhanced_matches = []

for _, unmatched_row in unmatched_speakers.head(30).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Split into words and clean
    unmatched_words = [word.strip() for word in unmatched.upper().split()]
    
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        available_words = [word.strip() for word in available.upper().split()]
        
        # Find common words
        common_words = set(unmatched_words).intersection(set(available_words))
        
        if len(common_words) > 0:
            # Different scoring for last names vs first names
            has_lastname_match = any(len(word) > 3 for word in common_words)  # Assume last names are longer
            word_coverage = len(common_words) / len(unmatched_words)
            
            # Prioritize matches with last names and good coverage
            if has_lastname_match and word_coverage >= 0.5:
                enhanced_matches.append({
                    'unmatched': unmatched,
                    'response_count': count,
                    'potential_match': available,
                    'common_words': ', '.join(sorted(common_words)),
                    'word_coverage': round(word_coverage, 3),
                    'confidence': 'HIGH' if word_coverage >= 0.7 else 'MEDIUM'
                })

# Sort by response count and confidence
enhanced_df = pd.DataFrame(enhanced_matches)
enhanced_df = enhanced_df.sort_values(['response_count', 'word_coverage'], ascending=[False, False])

print("Enhanced substring matches (focusing on high-confidence):")
print(enhanced_df.head(20))

Enhanced substring matches (focusing on high-confidence):
         unmatched  response_count                  potential_match  \
0   MARK COLERIDGE             127        Archbishop Mark Coleridge   
1   MARK COLERIDGE             127                       Mark Arbib   
2   MARK COLERIDGE             127                      Mark Bouris   
3   MARK COLERIDGE             127                      Mark Butler   
4   MARK COLERIDGE             127                    Mark Carnegie   
5   MARK COLERIDGE             127                         Mark Day   
6   MARK COLERIDGE             127                     Mark Dreyfus   
7   MARK COLERIDGE             127                      Mark Latham   
8   MARK COLERIDGE             127                     Mark Leibler   
9   MARK COLERIDGE             127                       Mark Scott   
10  MARK COLERIDGE             127                     Mark Seymour   
11  MARK COLERIDGE             127                       Mark Steyn   
12  MARK COLERIDGE 

In [23]:
# Look for obvious patterns
pattern_matches = []

for _, unmatched_row in unmatched_speakers.head(30).iterrows():
    unmatched = unmatched_row['unmatched_name']
    count = unmatched_row['response_count']
    
    # Check for exact substring containment
    for _, available_row in available_names.iterrows():
        available = available_row['available_name']
        
        # Check if unmatched is contained in available or vice versa
        unmatched_clean = unmatched.upper().replace('.', '').replace(',', '')
        available_clean = available.upper().replace('.', '').replace(',', '')
        
        if (unmatched_clean in available_clean or 
            available_clean in unmatched_clean or
            # Check reversed order (First Last vs Last First)
            ' '.join(reversed(unmatched_clean.split())) == available_clean):
            
            pattern_matches.append({
                'unmatched': unmatched,
                'response_count': count,
                'potential_match': available,
                'match_type': 'SUBSTRING/CONTAINMENT'
            })

pattern_df = pd.DataFrame(pattern_matches)
print("\nExact pattern matches:")
print(pattern_df.head(15))


Exact pattern matches:
           unmatched  response_count                  potential_match  \
0     MARK COLERIDGE             127        Archbishop Mark Coleridge   
1        PJ O'ROURKE             115                    P.J. O'Rourke   
2       SANDY GUTMAN             107  Sandy Gutman aka Austen Tayshus   
3        GEORGE PELL              80             Cardinal George Pell   
4   SUSAN GREENFIELD              68        Baroness Susan Greenfield   
5    JORDAN PETERSON              55               Dr Jordan Peterson   
6        CORNEL WEST              51                   Dr Cornel West   
7          CINDY PAN              49                     Dr Cindy Pan   
8    FELICITY HAMPEL              38            Judge Felicity Hampel   
9    FELICITY HAMPEL              38         Judge Felicity Hampel SC   
10   MOHAMAD ABDALLA              37               Dr Mohamad Abdalla   
11      MATT COLWELL              37             Matt Colwell aka 360   
12    PETER COSGROVE       